# 2주차 데이터 수집 파이프라인

## 목표
- 종목코드 검증 (pykrx 기준)
- 주요 20개 종목 일별 OHLCV 데이터 수집
- 외국인·기관 순매수 데이터 수집
- SQLite DB 적재

## 흐름
종목코드 검증 → OHLCV 수집 → 수급 데이터 수집 → SQLite 적재

## 종목 선정 기준
국내 핵심 5개 섹터(반도체, 플랫폼·통신, 방산, 에너지·전력, 로봇)별
대표 대형주 중심으로 4개씩 선정, 총 20개

In [5]:
import pandas as pd
from pykrx import stock
from datetime import datetime, timedelta
import sqlite3
import os

def get_safe_business_date(days_back=1):
    """
    안전한 영업일을 반환한다.
    - 월요일(weekday=0)이면 3일 전(금요일)로
    - 그 외에는 days_back일 전으로
    오늘이 장 마감 전이거나 휴장일일 수 있으므로
    최소 하루 전 날짜를 기본값으로 사용한다.
    """
    today = datetime.today()
    target = today - timedelta(days=days_back)
    
    # 토요일(5)이면 금요일로
    if target.weekday() == 5:
        target -= timedelta(days=1)
    # 일요일(6)이면 금요일로
    elif target.weekday() == 6:
        target -= timedelta(days=2)
    
    return target.strftime("%Y%m%d")

# 수집 기간 설정
END_DATE = get_safe_business_date(days_back=1)
START_DATE = (datetime.today() - timedelta(days=366)).strftime("%Y%m%d")

print(f"수집 기간: {START_DATE} ~ {END_DATE}")
print(f"검증 기준일: {END_DATE}")

수집 기간: 20250519 ~ 20260519
검증 기준일: 20260519


## 1단계 종목코드 검증

pykrx는 종목명이 아닌 종목코드(6자리)로 데이터를 조회한다.
사전에 정의한 종목코드가 pykrx에서 정상 조회되는지 확인한다.

두산로보틱스, 레인보우로보틱스 등 최근 상장 종목은
코드가 다를 수 있으므로 반드시 검증이 필요하다.

In [9]:
# 최종 확정 종목 딕셔너리
# 구조: {섹터: [(종목명, 종목코드), ...]}
STOCKS = {
    "반도체": [
        ("삼성전자", "005930"),
        ("SK하이닉스", "000660"),
        ("한미반도체", "042700"),
        ("리노공업", "058470"),
    ],
    "플랫폼·통신": [
        ("NAVER", "035420"),
        ("카카오", "035720"),
        ("KT", "030200"),
        ("SK텔레콤", "017670"),
    ],
    "방산": [
        ("한화에어로스페이스", "012450"),
        ("현대로템", "064350"),
        ("LIG넥스원", "079550"),
        ("한국항공우주", "047810"),
    ],
    "에너지·전력": [
        ("POSCO홀딩스", "005490"),
        ("LS ELECTRIC", "010120"),
        ("HD현대일렉트릭", "267260"),
        ("두산에너빌리티", "034020"),
    ],
    "로봇": [
        ("현대차", "005380"),
        ("HD현대", "267250"),
        ("두산로보틱스", "454910"),
        ("레인보우로보틱스", "277810"),
    ],
}

# pykrx에서 전체 종목 코드 목록 조회 (검증용)
# 날짜 명시적 지정 (코드셀 1에서 계산한 END_DATE 사용)
all_tickers = stock.get_market_ticker_list(date=END_DATE, market="ALL")

# 종목코드 검증
# get_market_ticker_list() 대신 실제 OHLCV 조회 성공 여부로 검증
# 이유: get_market_ticker_list()가 KRX 서버 응답 문제로 빈 값 반환

print("=== 종목코드 검증 결과 ===\n")
invalid = []

for sector, items in STOCKS.items():
    print(f"[{sector}]")
    for name, code in items:
        try:
            df = stock.get_market_ohlcv_by_date("20260519", "20260519", code)
            if df.empty:
                status = "오류 (빈 데이터)"
                invalid.append((name, code))
            else:
                status = "정상"
        except Exception as e:
            status = f"오류 ({e})"
            invalid.append((name, code))
        print(f"  {name} ({code}): {status}")
    print()

if invalid:
    print(f"오류 종목: {invalid}")
else:
    print("모든 종목코드 정상 확인")

Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
=== 종목코드 검증 결과 ===

[반도체]
  삼성전자 (005930): 정상
  SK하이닉스 (000660): 정상
  한미반도체 (042700): 정상
  리노공업 (058470): 정상

[플랫폼·통신]
  NAVER (035420): 정상
  카카오 (035720): 정상
  KT (030200): 정상
  SK텔레콤 (017670): 정상

[방산]
  한화에어로스페이스 (012450): 정상
  현대로템 (064350): 정상
  LIG넥스원 (079550): 정상
  한국항공우주 (047810): 정상

[에너지·전력]
  POSCO홀딩스 (005490): 정상
  LS ELECTRIC (010120): 정상
  HD현대일렉트릭 (267260): 정상
  두산에너빌리티 (034020): 정상

[로봇]
  현대차 (005380): 정상
  HD현대 (267250): 정상
  두산로보틱스 (454910): 정상
  레인보우로보틱스 (277810): 정상

모든 종목코드 정상 확인


## 2단계 OHLCV 데이터 수집

OHLCV란:
- O (Open):  시가 - 당일 시작 가격
- H (High):  고가 - 당일 최고 가격
- L (Low):   저가 - 당일 최저 가격
- C (Close): 종가 - 당일 마감 가격
- V (Volume): 거래량 - 당일 거래된 주식 수

pykrx의 get_market_ohlcv_by_date()로 종목별 일별 데이터를 수집한다.
수집 기간은 최근 1년(365일)으로 설정한다.

In [14]:
# 전체 종목 OHLCV 수집
ohlcv_list = []

for sector, items in STOCKS.items():
    for name, code in items:
        try:
            df = stock.get_market_ohlcv_by_date(START_DATE, END_DATE, code)
            df = df.reset_index()
            # 실제 반환 컬럼 기준으로 이름 지정
            # 날짜(인덱스) + 시가, 고가, 저가, 종가, 거래량, 등락률 = 7개
            df.columns = ["날짜", "시가", "고가", "저가", "종가", "거래량", "등락률"]
            df["종목코드"] = code
            df["종목명"] = name
            df["섹터"] = sector
            ohlcv_list.append(df)
            print(f"{name} ({code}): {len(df)}일치 수집 완료")
        except Exception as e:
            print(f"{name} ({code}): 수집 실패 → {e}")

# 전체 데이터 합치기
ohlcv_df = pd.concat(ohlcv_list, ignore_index=True)
print(f"\n전체 수집 완료: {len(ohlcv_df)}행")
print(ohlcv_df.head())

삼성전자 (005930): 245일치 수집 완료
SK하이닉스 (000660): 245일치 수집 완료
한미반도체 (042700): 245일치 수집 완료
리노공업 (058470): 245일치 수집 완료
NAVER (035420): 245일치 수집 완료
카카오 (035720): 245일치 수집 완료
KT (030200): 245일치 수집 완료
SK텔레콤 (017670): 245일치 수집 완료
한화에어로스페이스 (012450): 245일치 수집 완료
현대로템 (064350): 245일치 수집 완료
LIG넥스원 (079550): 245일치 수집 완료
한국항공우주 (047810): 245일치 수집 완료
POSCO홀딩스 (005490): 245일치 수집 완료
LS ELECTRIC (010120): 245일치 수집 완료
HD현대일렉트릭 (267260): 245일치 수집 완료
두산에너빌리티 (034020): 245일치 수집 완료
현대차 (005380): 245일치 수집 완료
HD현대 (267250): 245일치 수집 완료
두산로보틱스 (454910): 245일치 수집 완료
레인보우로보틱스 (277810): 245일치 수집 완료

전체 수집 완료: 4900행
          날짜     시가     고가     저가     종가       거래량       등락률    종목코드   종목명  \
0 2025-05-19  56400  56400  55500  55800   9802105 -1.760563  005930  삼성전자   
1 2025-05-20  56200  56700  55700  55900   9080577  0.179211  005930  삼성전자   
2 2025-05-21  56200  56600  55700  55700   7794181 -0.357782  005930  삼성전자   
3 2025-05-22  55300  55500  54500  54700  15254278 -1.795332  005930  삼성전자   
4 2025-05-23  55000

## 3단계 외국인·기관 순매수 데이터 수집

순매수란:
- 매수 금액 - 매도 금액의 차이
- 양수(+): 순매수 → 해당 주체가 더 많이 샀다
- 음수(-): 순매도 → 해당 주체가 더 많이 팔았다

외국인·기관의 순매수 동향은 수급 분석의 핵심 지표다.
pykrx의 get_market_trading_value_by_date()로 수집한다.

In [30]:
# 수급 데이터 수집 보류
# 원인: KRX 서버 해당 엔드포인트 응답 없음 (pykrx 함수 정상, 서버 문제)
# 대응: 3주차 분석 모델링 전 재시도 예정
# 현재 단계에서는 OHLCV 데이터만으로 파이프라인 검증 진행

print("수급 데이터 수집 보류 — KRX 서버 응답 없음")
print("재시도 예정: 3주차 분석 모델링 단계")
print("현재 단계: OHLCV 데이터로 파이프라인 검증 진행")

수급 데이터 수집 보류 — KRX 서버 응답 없음
재시도 예정: 3주차 분석 모델링 단계
현재 단계: OHLCV 데이터로 파이프라인 검증 진행


## 4단계 SQLite DB 적재

SQLite란:
- 파일 하나(.db)로 동작하는 가벼운 데이터베이스
- 서버 설치 없이 로컬에서 바로 사용 가능
- MVP 단계에서 가장 적합한 선택

테이블 구조:
- ohlcv: 종목별 일별 OHLCV 데이터
- supply: 종목별 일별 외국인·기관 순매수 데이터

In [32]:
# DB 파일 경로 설정
DB_PATH = "../database/stock_data.db"
os.makedirs("../database", exist_ok=True)

# DB 연결
conn = sqlite3.connect(DB_PATH)

# OHLCV 테이블 적재
ohlcv_df.to_sql("ohlcv", conn, if_exists="replace", index=False)
print(f"ohlcv 테이블 적재 완료: {len(ohlcv_df)}행")

# 수급 테이블은 KRX 서버 문제로 보류
# supply_df.to_sql("supply", conn, if_exists="replace", index=False)

conn.close()
print(f"\nDB 저장 완료: {DB_PATH}")

ohlcv 테이블 적재 완료: 4900행

DB 저장 완료: ../database/stock_data.db


## 5단계 적재 결과 검증

DB에 정상적으로 적재되었는지 조회로 확인한다.

확인 항목:
- [ ] ohlcv 테이블 행 수가 예상치와 일치하는가
- [ ] supply 테이블 행 수가 예상치와 일치하는가
- [ ] 종목별 데이터가 빠짐없이 들어갔는가

In [33]:
# DB 재연결 후 검증
conn = sqlite3.connect(DB_PATH)

# ohlcv 테이블 확인
ohlcv_check = pd.read_sql(
    "SELECT 종목명, COUNT(*) as 수집일수 FROM ohlcv GROUP BY 종목명", conn
)
print("=== OHLCV 테이블 검증 ===")
print(ohlcv_check.to_string(index=False))

conn.close()

=== OHLCV 테이블 검증 ===
        종목명  수집일수
       HD현대   245
   HD현대일렉트릭   245
         KT   245
     LIG넥스원   245
LS ELECTRIC   245
      NAVER   245
   POSCO홀딩스   245
      SK텔레콤   245
     SK하이닉스   245
     두산로보틱스   245
    두산에너빌리티   245
   레인보우로보틱스   245
       리노공업   245
       삼성전자   245
        카카오   245
     한국항공우주   245
      한미반도체   245
  한화에어로스페이스   245
       현대로템   245
        현대차   245
